In [ ]:
# 1) Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_validate, StratifiedKFold)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, RocCurveDisplay)

In [ ]:
# 2) Load dataset
# Source: Merged Cleveland + Statlog + Hungarian Heart Disease datasets
# Kaggle: https://www.kaggle.com/datasets/sid321axn/heart-statlog-cleveland-hungary-final
df = pd.read_csv("heart_statlog_cleveland_hungary_final.xls")
print("Shape:", df.shape)
df.head()

In [ ]:
# 3) Basic structure checks
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)

In [ ]:
# 4) Check missing values and duplicates
print("Missing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# 5) Remove exact duplicate rows
df_clean = df.drop_duplicates().reset_index(drop=True)

print("Original shape :", df.shape)
print("Cleaned shape  :", df_clean.shape)
print("Rows removed   :", df.shape[0] - df_clean.shape[0])
print("Remaining duplicates:", df_clean.duplicated().sum())

In [ ]:
# 6) Handle zero cholesterol — physiologically impossible, treat as missing
#    Replace 0s with NaN, then impute with median (robust to skew)
print("Zero cholesterol rows (before):", (df_clean['cholesterol'] == 0).sum())
print("Zero resting bp s rows        :", (df_clean['resting bp s'] == 0).sum())

df_clean['cholesterol']  = df_clean['cholesterol'].replace(0, np.nan)
df_clean['resting bp s'] = df_clean['resting bp s'].replace(0, np.nan)

# Impute with median
for col in ['cholesterol', 'resting bp s']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"Imputed '{col}' NaNs with median = {median_val:.1f}")

print("\nMissing values after imputation:\n", df_clean.isna().sum())

In [ ]:
# 7) Define target and feature groups
target_col  = "target"
continuous  = ["age", "resting bp s", "cholesterol", "max heart rate", "oldpeak"]
categorical = ["sex", "chest pain type", "fasting blood sugar",
               "resting ecg", "exercise angina", "ST slope"]

print("Target        :", target_col)
print("Continuous    :", continuous)
print("Categorical   :", categorical)

In [ ]:
# 8) Confirm unique values for categorical features
for col in categorical:
    print(col, "->", sorted(df_clean[col].unique()))

In [ ]:
# 9) Target distribution plot
counts = df_clean[target_col].value_counts().sort_index()
print(counts)

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index.astype(str), counts.values,
               color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)
plt.title("Target Distribution (0 = No Disease, 1 = Disease)", fontsize=13, fontweight='bold')
plt.xlabel("Target")
plt.ylabel("Count")

total = counts.sum()
for bar in bars:
    height = bar.get_height()
    pct = (height / total) * 100
    plt.text(bar.get_x() + bar.get_width() / 2, height + 3,
             f"{int(height)}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 10) Summary statistics — continuous features (before scaling)
print("=== Summary Statistics — Raw (Before Scaling) ===")
df_clean[continuous].describe().T

In [ ]:
# 11) Correlation heatmap — continuous features + target
corr = df_clean[continuous + [target_col]].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
plt.imshow(corr.values, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.values[i,j]:.2f}",
                 ha="center", va="center", fontsize=8)
plt.title("Correlation Heatmap (Continuous Features + Target)")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# 12) Histograms — continuous features
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

feat_labels = {
    "age":            "Age (years)",
    "resting bp s":   "Resting Blood Pressure (mmHg)",
    "cholesterol":    "Cholesterol (mg/dl)",
    "max heart rate": "Max Heart Rate (bpm)",
    "oldpeak":        "ST Depression (oldpeak)"
}
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']

for i, (col, color) in enumerate(zip(continuous, colors)):
    ax = axes[i]
    data = df_clean[col]
    ax.hist(data, bins=25, color=color, edgecolor='white', linewidth=0.6, alpha=0.90)
    ax.axvline(data.mean(),   color='black',  linestyle='--', linewidth=1.4,
               label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='-',  linewidth=1.4,
               label=f'Median: {data.median():.1f}')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(feat_labels[col], fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(fontsize=8, framealpha=0.7)

axes[-1].set_visible(False)
fig.suptitle("Histograms — Continuous Features (Before Scaling)",
             fontsize=14, fontweight='bold', y=1.01)
plt.subplots_adjust(hspace=0.50, wspace=0.35)
plt.show()

In [ ]:
# 13) Categorical feature distributions vs target
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

cat_labels = {
    "sex":                 ["Female", "Male"],
    "chest pain type":     ["Type 1", "Type 2", "Type 3", "Type 4"],
    "fasting blood sugar": ["≤120 mg/dl", ">120 mg/dl"],
    "resting ecg":         ["Normal", "ST-T abnorm.", "LV hypertrophy"],
    "exercise angina":     ["No", "Yes"],
    "ST slope":            ["Unspecified", "Upsloping", "Flat", "Downsloping"]
}

for i, col in enumerate(categorical):
    ax = axes[i]
    ct = df_clean.groupby([col, target_col]).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452'],
            edgecolor='white', linewidth=0.6)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(["No Disease", "Disease"], fontsize=8)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle("Categorical Features vs Target", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 14) Split X and y
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]
print("X shape:", X.shape, "| y shape:", y.shape)
print("\nClass distribution:\n", y.value_counts())

In [ ]:
# 15) Train-test split — stratified 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train class dist:\n", y_train.value_counts())
print("Test class dist:\n",  y_test.value_counts())

## Feature Scaling

`StandardScaler` is fitted **only on the training set** to prevent data leakage, then applied to both splits.  
This explicit step is used for EDA visualisations only.  
The model pipelines (Cell 17+) contain their own internal scaler — **no double-scaling occurs**.

In [ ]:
# 16) Explicit feature scaling — fit on train, transform both (EDA only)
scaler = StandardScaler()
X_train_cont_scaled = scaler.fit_transform(X_train[continuous])
X_test_cont_scaled  = scaler.transform(X_test[continuous])

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[continuous] = X_train_cont_scaled
X_test_scaled[continuous]  = X_test_cont_scaled

print("Scaling fitted on TRAIN set only — no data leakage.")
print("\n=== Scaled Continuous Features — Train (first 5 rows) ===")
print(X_train_scaled[continuous].head().to_string())
print("\n=== Summary Statistics — Scaled Continuous (Train) ===")
print(pd.DataFrame(X_train_cont_scaled, columns=continuous).describe().T.round(4).to_string())

In [ ]:
# 17) Before vs After scaling — side-by-side comparison
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, col in enumerate(continuous):
    axes[0, i].hist(X_train[col], bins=25, color="steelblue", edgecolor="white")
    axes[0, i].set_title(f"{col}\n(raw)", fontsize=9)
    axes[1, i].hist(X_train_cont_scaled[:, i], bins=25, color="darkorange", edgecolor="white")
    axes[1, i].set_title(f"{col}\n(scaled)", fontsize=9)
axes[0, 0].set_ylabel("Before Scaling", fontweight="bold")
axes[1, 0].set_ylabel("After Scaling",  fontweight="bold")
plt.suptitle("Continuous Features: Before vs After StandardScaler (Train Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 18) Preprocessing pipeline (used INSIDE each model pipeline)
#     Pipeline fits its own scaler on each CV train fold — leakage-free
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), continuous),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)

In [ ]:
# 19) Build baseline model pipelines
models = {
    "LogReg":       LogisticRegression(max_iter=2000, random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "GradBoost":    GradientBoostingClassifier(random_state=42),
    "SVC":          SVC(probability=True, random_state=42),
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    for name, model in models.items()
}

In [ ]:
# 20) Evaluate all models — 5-Fold Stratified CV + held-out test set
#     Using the same StratifiedKFold object for consistency
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, pipe in pipelines.items():
    # --- 5-Fold CV on training set ---
    cv_scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=["accuracy", "recall", "f1", "roc_auc"],
        return_train_score=False
    )

    # --- Final evaluation on held-out test set ---
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model":          name,
        "CV Accuracy":    f"{cv_scores['test_accuracy'].mean():.4f} ± {cv_scores['test_accuracy'].std():.4f}",
        "CV Recall":      f"{cv_scores['test_recall'].mean():.4f} ± {cv_scores['test_recall'].std():.4f}",
        "CV F1":          f"{cv_scores['test_f1'].mean():.4f} ± {cv_scores['test_f1'].std():.4f}",
        "CV ROC-AUC":     f"{cv_scores['test_roc_auc'].mean():.4f} ± {cv_scores['test_roc_auc'].std():.4f}",
        "Test Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Test Recall":    round(recall_score(y_test, y_pred), 4),
        "Test F1":        round(f1_score(y_test, y_pred), 4),
        "Test ROC-AUC":   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).sort_values(by="Test ROC-AUC", ascending=False)
print("=== Baseline Model Comparison — 5-Fold CV + Held-out Test ===")
results_df

In [ ]:
# 21) Hyperparameter tuning — Logistic Regression (GridSearchCV, 5-Fold CV)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

param_grid = {
    "model__C":       [0.01, 0.1, 1, 10, 100],
    "model__solver":  ["lbfgs"],
    "model__penalty": ["l2"]
}

grid = GridSearchCV(
    logreg_pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best parameters :", grid.best_params_)
print("Best CV ROC-AUC :", round(grid.best_score_, 4))
print("Train ROC-AUC   :", round(
    grid.cv_results_['mean_train_score'][grid.best_index_], 4))

In [ ]:
# 22) Evaluate tuned Logistic Regression on held-out test set
best_model = grid.best_estimator_

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("=== Tuned Logistic Regression — Test Set Performance ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred),   4))
print("F1       :", round(f1_score(y_test, y_pred),       4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_proba), 4))

# Confusion matrix
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["No Disease", "Disease"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Tuned Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
# 23) ROC curve — Tuned Logistic Regression
RocCurveDisplay.from_predictions(
    y_test, y_proba, name="Tuned Logistic Regression"
)
plt.title("ROC Curve — Tuned Logistic Regression")
plt.tight_layout()
plt.show()

In [ ]:
# 24) Feature importance — coefficient plot (Logistic Regression)
#     Extract feature names after OneHotEncoding
ohe_features = (best_model.named_steps['preprocess']
                .named_transformers_['cat']
                .get_feature_names_out(categorical).tolist())
all_features  = continuous + ohe_features
coefficients  = best_model.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'Feature':     all_features,
    'Coefficient': coefficients
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 7))
colors_bar = ['#DD8452' if c > 0 else '#4C72B0' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'],
         color=colors_bar, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title("Logistic Regression — Feature Coefficients (Tuned Model)",
          fontsize=13, fontweight='bold')
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

print("\nTop 5 positive predictors (increase disease risk):")
print(coef_df.head(5).to_string(index=False))
print("\nTop 5 negative predictors (decrease disease risk):")
print(coef_df.tail(5).to_string(index=False))

In [ ]:
# 25) Save final tuned pipeline model
joblib.dump(best_model, "heart_disease_model.joblib")
print("Saved: heart_disease_model.joblib")

In [ ]:
# 26) Sample prediction — single patient from test set
sample = X_test.iloc[[0]]
pred   = best_model.predict(sample)[0]
proba  = best_model.predict_proba(sample)[0, 1]

print("Patient features:")
print(sample.T.to_string())
print("\nPrediction  :", pred, " (0 = No Disease, 1 = Disease)")
print("Probability of Disease:", round(proba, 4))